# Partie 1 — Bronze : Spark + PostgreSQL + CSV + JSON + Iceberg

**TP pratique — version complète et exécutable**

Dans ce notebook, nous allons créer les trois tables de la couche **Bronze** :

- `medical.bronze.patients` : patients lus dans **PostgreSQL** ;
- `medical.bronze.consultations` : consultations lues dans **encounters.csv** ;
- `medical.bronze.observations` : observations lues dans **observations.jsonl**.

**Bronze** conserve les données sources : pas de nettoyage, pas de suppression de doublons, pas de jointures. Le nettoyage sera effectué dans **Silver**.

**Avant de commencer :** les conteneurs doivent être démarrés, le bucket `medical-warehouse` doit exister dans RustFS et la table PostgreSQL `patients_source` doit déjà être remplie. Nous avons précédemment importé **108 patients**.

**Exécution :** dans JupyterLab, ouvrez ce notebook, choisissez **Kernel → Restart Kernel and Run All Cells**. Vous n'avez pas besoin d'ouvrir l'ancien notebook. Si les tables Bronze existent déjà, ce notebook les **conserve** et ne rajoute pas de doublons.

> Identifiants PostgreSQL (`user` / `0000`) : uniquement pour ce laboratoire local, à ne pas réutiliser en production.

## 1 — Démarrer une nouvelle session Spark

Spark exécute les traitements. Nous le configurons pour accéder au **catalogue Iceberg REST** (`iceberg-rest`) et au **stockage RustFS** (`rustfs`).

- `medical` : nom de notre catalogue Iceberg ;
- `medical-warehouse` : bucket de stockage des tables ;
- `local[2]` : deux cœurs de calcul dans le conteneur.

In [ ]:
# 1. Créer la session Spark de ce notebook
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .master("local[2]")
    .appName("MedicalLakehouse-Partie1-Bronze")
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions")
    .config("spark.sql.catalog.medical", "org.apache.iceberg.spark.SparkCatalog")
    .config("spark.sql.catalog.medical.type", "rest")
    .config("spark.sql.catalog.medical.uri", "http://iceberg-rest:8181")
    .config("spark.sql.catalog.medical.warehouse", "s3://medical-warehouse/")
    .config("spark.sql.catalog.medical.io-impl", "org.apache.iceberg.aws.s3.S3FileIO")
    .config("spark.sql.catalog.medical.s3.endpoint", "http://rustfs:9000")
    .config("spark.sql.catalog.medical.s3.path-style-access", "true")
    .getOrCreate()
)

spark.sparkContext.setLogLevel("ERROR")
print("Version de Spark :", spark.version)
print("Catalogue :", spark.conf.get("spark.sql.catalog.medical"))
print("REST :", spark.conf.get("spark.sql.catalog.medical.uri"))

## 2 — Vérifier le catalogue Iceberg

Le catalogue permet de retrouver les tables ; les fichiers seront enregistrés dans RustFS. Nous testons d'abord cette connexion avant de charger les données.

In [ ]:
# 2. Vérifier que Spark voit le catalogue
spark.sql("SHOW NAMESPACES IN medical").show(truncate=False)

## 3 — Vérifier les fichiers Synthea

Les fichiers sont sur Windows/WSL dans le dossier `medical-lakehouse/notebooks/data/`, visible dans le conteneur sous `/workspace/notebooks/data/` grâce au volume Docker.

In [ ]:
# 3. Vérifier les chemins des fichiers
from pathlib import Path

DATA = Path("/workspace/notebooks/data")
SYNTHEA = DATA / "synthea"
PREPARED = DATA / "prepared"

patients_csv = SYNTHEA / "patients.csv"   # archive d'origine (non utilisée pour Bronze)
consultations_csv = SYNTHEA / "encounters.csv"
observations_csv = SYNTHEA / "observations.csv"
observations_json = PREPARED / "observations.jsonl"

for chemin in [patients_csv, consultations_csv, observations_csv]:
    print(chemin.name, "→", "OK" if chemin.is_file() else "INTROUVABLE")
    if not chemin.is_file():
        raise FileNotFoundError(f"Fichier manquant : {chemin}")

PREPARED.mkdir(parents=True, exist_ok=True)

## 4 — Préparer `observations.jsonl`

Synthea fournit les observations en CSV. Pour simuler une troisième source, nous créons **un objet JSON par ligne** en conservant les valeurs d'origine. Cette préparation ne constitue pas un nettoyage Silver.

In [ ]:
# 4. Convertir le CSV d'observations en JSON Lines
import csv
import json

nombre_converti = 0

with observations_csv.open("r", encoding="utf-8-sig", newline="") as fichier_csv:
    lecteur = csv.DictReader(fichier_csv)
    with observations_json.open("w", encoding="utf-8") as fichier_json:
        for ligne in lecteur:
            fichier_json.write(json.dumps(ligne, ensure_ascii=False) + "\n")
            nombre_converti += 1

print("Fichier JSON créé :", observations_json)
print("Nombre d'observations converties :", nombre_converti)

## 5 — Lire les patients depuis PostgreSQL

Nous ne réutilisons **aucune variable** d'un autre notebook : la connexion JDBC et le DataFrame sont créés ici.

À l'intérieur de Docker, PostgreSQL est accessible sur `postgres:5432`, et non sur `localhost:5433`.

In [ ]:
# 5. Paramètres de connexion à PostgreSQL
jdbc_options = {
    "url": "jdbc:postgresql://postgres:5432/hospital",
    "dbtable": "public.patients_source",
    "user": "user",
    "password": "0000",
    "driver": "org.postgresql.Driver"
}

# Lire les patients directement depuis PostgreSQL
df_patients = (
    spark.read
    .format("jdbc")
    .options(**jdbc_options)
    .load()
)

df_patients.printSchema()
df_patients.show(5, truncate=False)
nb_patients = df_patients.count()
print("Patients PostgreSQL :", nb_patients)
assert nb_patients > 0, "La table PostgreSQL patients_source est vide."

## 6 — Lire les consultations depuis le CSV

Nous conservons les colonnes et les valeurs sources. Les dates restent du texte dans Bronze.

In [ ]:
# 6. Lire les consultations depuis encounters.csv
df_consultations = (
    spark.read
    .option("header", True)
    .option("inferSchema", False)
    .csv(str(consultations_csv))
)

df_consultations.printSchema()
df_consultations.show(5, truncate=False)
nb_consultations = df_consultations.count()
print("Consultations CSV :", nb_consultations)
assert nb_consultations > 0, "Le fichier encounters.csv semble vide."

## 7 — Lire les observations depuis le JSON

Spark lit les objets JSON ligne par ligne et construit un DataFrame. Nous ne transformons pas les unités ni les valeurs médicales à cette étape.

In [ ]:
# 7. Lire les observations en JSON
df_observations = spark.read.json(str(observations_json))

df_observations.printSchema()
df_observations.show(5, truncate=False)
nb_observations = df_observations.count()
print("Observations JSON :", nb_observations)
assert nb_observations == nombre_converti, "Le nombre de lignes JSON est différent du CSV."

## 8 — Vérifier les trois sources avant l'ingestion

Les quantités affichées servent de référence : Bronze devra contenir **autant de lignes que les sources**.

In [ ]:
# 8. Récapitulatif des sources
print("=== SOURCES ===")
print("Patients PostgreSQL :", nb_patients)
print("Consultations CSV   :", nb_consultations)
print("Observations JSON   :", nb_observations)

## 9 — Créer le namespace `bronze`

Le nom `medical.bronze.patients` signifie : **catalogue `medical` → namespace `bronze` → table `patients`**.

`IF NOT EXISTS` permet de relancer la cellule sans erreur si le namespace a déjà été créé.

In [ ]:
# 9. Créer l'espace Bronze dans le catalogue
spark.sql("CREATE NAMESPACE IF NOT EXISTS medical.bronze")
spark.sql("SHOW NAMESPACES IN medical").show(truncate=False)

## 10 — Écrire la table Bronze des patients

Spark copie les lignes de PostgreSQL dans une **table Iceberg**. Iceberg référence des fichiers de données et leurs métadonnées dans RustFS.

Si la table existe déjà, nous **ne l'écrasons pas** et ne dupliquons pas les lignes.

In [ ]:
# 10. PostgreSQL -> Iceberg Bronze patients
nom_table = "medical.bronze.patients"

if spark.catalog.tableExists(nom_table):
    print("Table déjà existante, aucune nouvelle écriture :", nom_table)
else:
    (
        df_patients.writeTo(nom_table)
        .using("iceberg")
        .create()
    )
    print("Table créée :", nom_table)

print("Patients dans Bronze :", spark.table(nom_table).count())
spark.table(nom_table).show(5, truncate=False)

## 11 — Écrire la table Bronze des consultations

Même opération, mais cette fois les données viennent d'un fichier CSV. Les champs d'origine sont conservés.

In [ ]:
# 11. CSV -> Iceberg Bronze consultations
nom_table = "medical.bronze.consultations"

if spark.catalog.tableExists(nom_table):
    print("Table déjà existante, aucune nouvelle écriture :", nom_table)
else:
    (
        df_consultations.writeTo(nom_table)
        .using("iceberg")
        .create()
    )
    print("Table créée :", nom_table)

print("Consultations dans Bronze :", spark.table(nom_table).count())
spark.table(nom_table).show(5, truncate=False)

## 12 — Écrire la table Bronze des observations

Les observations JSON deviennent à leur tour une table Iceberg. Nous ne nettoyons pas encore les valeurs.

In [ ]:
# 12. JSON -> Iceberg Bronze observations
nom_table = "medical.bronze.observations"

if spark.catalog.tableExists(nom_table):
    print("Table déjà existante, aucune nouvelle écriture :", nom_table)
else:
    (
        df_observations.writeTo(nom_table)
        .using("iceberg")
        .create()
    )
    print("Table créée :", nom_table)

print("Observations dans Bronze :", spark.table(nom_table).count())
spark.table(nom_table).show(5, truncate=False)

## 13 — Afficher les tables du catalogue

`SHOW TABLES` vérifie que le catalogue Iceberg connaît nos trois tables Bronze.

In [ ]:
# 13. Lister les tables de la couche Bronze
spark.sql("SHOW TABLES IN medical.bronze").show(truncate=False)

## 14 — Vérifier le nombre de lignes

Bronze doit conserver toutes les lignes. Si une table existait déjà avec un autre contenu, un message **ATTENTION** s'affichera : elle ne sera pas modifiée automatiquement.

In [ ]:
# 14. Comparer les nombres de lignes sources et Bronze
comparaison = [
    ("patients", nb_patients),
    ("consultations", nb_consultations),
    ("observations", nb_observations)
]

for nom, nb_source in comparaison:
    nb_bronze = spark.table(f"medical.bronze.{nom}").count()
    etat = "OK" if nb_source == nb_bronze else "ATTENTION : nombres différents"
    print(f"{nom} : source={nb_source}, bronze={nb_bronze} -> {etat}")

## 15 — Interroger Bronze avec Spark SQL

Les données sont maintenant consultables comme des tables SQL, même si elles sont physiquement stockées dans RustFS.

In [ ]:
# 15. Exemple de requête SQL sur Bronze
spark.sql("""
    SELECT patient_id, birthdate, gender, city
    FROM medical.bronze.patients
    LIMIT 5
""").show(truncate=False)

## 16 — Voir les fichiers de la table Iceberg

La table de métadonnées `.files` contient notamment le chemin de chaque fichier de données et son nombre de lignes. Vous devriez voir des chemins vers le bucket `medical-warehouse`.

In [ ]:
# 16. Afficher les fichiers de données Iceberg
spark.sql("""
    SELECT file_path, record_count, file_size_in_bytes
    FROM medical.bronze.patients.files
""").show(truncate=False)

## 17 — Voir les snapshots Iceberg

Un **snapshot** correspond à un état enregistré de la table. Iceberg utilise ces snapshots pour suivre les changements de données.

In [ ]:
# 17. Consulter l'historique des snapshots
spark.sql("""
    SELECT snapshot_id, committed_at, operation
    FROM medical.bronze.patients.snapshots
""").show(truncate=False)

## 18 — Vérifier les objets dans RustFS

1. Ouvrir [RustFS Console](http://localhost:9001).
2. Accéder au bucket **`medical-warehouse`**.
3. Parcourir les dossiers créés par Iceberg ; les chemins exacts peuvent varier.
4. Repérer les fichiers **Parquet** et les fichiers de **métadonnées Iceberg**.

**Ne pas déplacer, renommer ou supprimer manuellement ces fichiers** : ils sont gérés par Iceberg.

## Bilan — Fin de la partie Bronze

Vous disposez maintenant de trois tables Iceberg :

```text
medical
└── bronze
    ├── patients       ← PostgreSQL
    ├── consultations  ← CSV
    └── observations   ← JSON
```

**Résultats attendus :** trois tables visibles dans le catalogue, des nombres de lignes identiques aux sources, des fichiers physiques dans RustFS et des snapshots consultables.

**Prochaine partie : Silver** — conversion des dates, nettoyage, contrôles de qualité et relations entre patients, consultations et observations.

> **En cas d'erreur :** exécuter les cellules depuis le début. Si l'erreur indique `NameError`, une cellule précédente n'a pas été exécutée. Si une table existe déjà, le notebook ne l'écrase pas. Si l'erreur concerne S3 ou Iceberg, vérifier les services Docker et le bucket `medical-warehouse`.